In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [ ]:
# ==========================================
# 데이터 로드
# ==========================================
# 데이터 파일명은 실제 환경에 맞게 수정해주세요.
df = pd.read_csv('Customer_Churn.csv')
print("데이터 로드 완료. 원본 데이터 크기:", df.shape)


데이터 로드 완료. 원본 데이터 크기: (7043, 21)


In [ ]:
# ==========================================
# 기본 전처리 및 결측치 처리
# ==========================================
# 불필요한 고유 식별자 피처 제거
df = df.drop('customerID', axis=1)

# TotalCharges 열의 공백 결측치를 0으로 변환 후 숫자형으로 캐스팅
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [ ]:
# ==========================================
# 핵심 피처 생성 (Feature Engineering)
# ==========================================
# 부가 서비스 관여도 (Service Engagement Score)
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Service_Engagement'] = 0
for service in services:
    df['Service_Engagement'] += (df[service] == 'Yes').astype(int)

# 가입 기간 범주화 (Tenure Group)
def tenure_to_group(tenure):
    if tenure <= 12:
        return 'New(0-1yr)'
    elif tenure <= 36:
        return 'Standard(1-3yr)'
    else:
        return 'Loyal(3yr+)'
df['Tenure_Group'] = df['tenure'].apply(tenure_to_group)


In [ ]:
# ==========================================
# 범주형 데이터 인코딩
# ==========================================
# 이진형 데이터 변환 (Label Encoding)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 다중 클래스 범주 데이터 변환 (One-Hot Encoding)
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod', 'Tenure_Group']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)


In [ ]:
# ==========================================
# 상관관계 분석
# ==========================================
corr_with_churn = df.corr()['Churn'].sort_values(ascending=False)
print("\n=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').head(5))

print("\n=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').tail(5))



=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===
Tenure_Group_New(0-1yr)           0.317580
InternetService_Fiber optic       0.308020
PaymentMethod_Electronic check    0.301919
MonthlyCharges                    0.193356
PaperlessBilling                  0.191825
Name: Churn, dtype: float64

=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===
TechSupport_No internet service        -0.227890
DeviceProtection_No internet service   -0.227890
StreamingTV_No internet service        -0.227890
Contract_Two year                      -0.302253
tenure                                 -0.352229
Name: Churn, dtype: float64


In [ ]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.48      0.56       373

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.80      0.79      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1720)
2. MonthlyCharges (0.1593)
3. tenure (0.1446)
4. Service_Engagement (0.0406)
5. Tenure_Group_New(0-1yr) (0.0387)


In [ ]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.65      0.45      0.53       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1551)
2. tenure (0.1506)
3. MonthlyCharges (0.1449)
4. Contract_Two year (0.0469)
5. InternetService_Fiber optic (0.0421)


In [ ]:
# 성능 지표 이후 성능 개선 방법
# 피처 엔지니어링: 가입 기간 대비 평균 납부액 추가
# df는 기존 데이터프레임이라고 가정합니다.
df['Avg_Monthly_Spent'] = df['TotalCharges'] / (df['tenure'] + 1) # 0으로 나누기 방지 (+1)

In [ ]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1286)
2. tenure (0.1277)
3. MonthlyCharges (0.1205)
4. Avg_Monthly_Spent (0.1198)
5. Contract_Two year (0.0577)


In [ ]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 임계값 조정 (Threshold Adjustment)
# predict() 대신 predict_proba()를 사용하여 클래스 1(이탈)에 대한 확률값을 가져옵니다.
y_probs = rf_model.predict_proba(X_test)[:, 1]

# 임계값을 0.3으로 설정 (0.5보다 낮게 설정하여 이탈자를 더 공격적으로 탐색)
custom_threshold = 0.3
y_pred_adjusted = (y_probs >= custom_threshold).astype(int)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. TotalCharges (0.1286)
2. tenure (0.1277)
3. MonthlyCharges (0.1205)
4. Avg_Monthly_Spent (0.1198)
5. Contract_Two year (0.0577)


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:


# 피처 엔지니어링: 가입 기간 대비 평균 납부액 추가
# df는 기존 데이터프레임이라고 가정합니다.
df['Avg_Monthly_Spent'] = df['TotalCharges'] / (df['tenure'] + 1) # 0으로 나누기 방지 (+1)

# 데이터 분리 (X: 피처, y: 타겟)
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 모델 교체: XGBoost 사용 및 불균형 가중치(scale_pos_weight) 설정
# scale_pos_weight = (음성 샘플 수 / 양성 샘플 수) 대략 3 정도로 설정
model_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,    # 5 -> 3,4 과적합 방지
    scale_pos_weight=4,  # 이탈 데이터(1)에 약 3배의 가중치 부여
    #min_child_weight=30,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)

# 임계값(Threshold) 조정: 0.5 -> 0.3
# 단순히 predict()가 아니라 predict_proba()를 사용해 확률값을 가져옵니다.
y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]
custom_threshold = 0.5
y_pred_custom = (y_pred_proba >= custom_threshold).astype(int)

# 성능 지표 출력
print(f"=== [임계값 {custom_threshold} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))

=== [임계값 0.5 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.93      0.66      0.77      1035
           1       0.48      0.86      0.61       374

    accuracy                           0.72      1409
   macro avg       0.70      0.76      0.69      1409
weighted avg       0.81      0.72      0.73      1409



c:\python_proj\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:25:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import pandas as pd

# 기존 테스트 데이터에 이탈 확률 컬럼 추가
# X_test에 인덱스가 유지되어 있다고 가정합니다.
analysis_df = X_test.copy()
analysis_df['Churn_Prob'] = y_probs  # 모델이 계산한 확률값 (0~1)
analysis_df['Actual_Churn'] = y_test # 실제 이탈 여부 (비교용)

# 2. 확률 구간(Grade) 나누기 (0~100%를 5개 구간으로 구분)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
analysis_df['Prob_Grade'] = pd.cut(analysis_df['Churn_Prob'], bins=bins, labels=labels)

# 3. 그룹별 기초 통계 지표 추출
# 분석하고 싶은 주요 컬럼들을 선정합니다.
metrics = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Avg_Monthly_Spent', 'Actual_Churn']

# 그룹별 평균값 계산
group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()

# 그룹별 고객 수(Count) 추가
group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()

print("=== [이탈 확률 구간별 기초 통계 분석] ===")
display(group_stats)

# 4. 특정 고위험군(Very High) 고객 리스트만 따로 뽑기
high_risk_customers = analysis_df[analysis_df['Prob_Grade'] == 'Very High'].sort_values(by='Churn_Prob', ascending=False)

print(f"\n=== [초고위험군(80% 이상) 샘플 5건] ===")
display(high_risk_customers.head(5))

=== [이탈 확률 구간별 기초 통계 분석] ===


C:\Users\LG\AppData\Local\Temp\ipykernel_4688\554570965.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()
C:\Users\LG\AppData\Local\Temp\ipykernel_4688\554570965.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()


,tenure,MonthlyCharges,TotalCharges,Avg_Monthly_Spent,Actual_Churn,Customer_Count
Prob_Grade,,,,,,
Very Low,32.908007,62.866525,2239.250426,57.539717,0.236797,587
Low,29.515789,63.798246,2070.235965,57.425899,0.284211,285
Medium,34.175000,66.123250,2361.060000,60.596516,0.265000,200
High,33.447154,64.181707,2275.234146,58.148198,0.252033,123
Very High,30.719298,67.735088,2305.677193,61.182048,0.315789,57



=== [초고위험군(80% 이상) 샘플 5건] ===


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Service_Engagement,...,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr),Avg_Monthly_Spent,Churn_Prob,Actual_Churn,Prob_Grade
5006,0,0,0,0,15,1,1,55.70,899.80,1,...,False,False,False,True,False,True,56.237500,1.00,1,Very High
5444,1,0,1,0,45,1,1,107.75,4882.80,4,...,False,False,False,False,False,False,106.147826,1.00,0,Very High
4512,0,1,0,0,70,1,1,75.50,5212.65,4,...,True,False,False,False,False,False,73.417606,1.00,0,Very High
6346,1,1,1,1,1,1,1,49.25,49.25,1,...,False,True,False,False,True,False,24.625000,0.99,1,Very High
4379,0,0,1,0,68,1,1,89.95,5974.30,2,...,True,False,False,False,False,False,86.584058,0.99,0,Very High


In [ ]:
# 분석용 데이터프레임 복사 및 확률 추가
action_df = X_test.copy()
action_df['Churn_Prob'] = y_probs  # 모델의 예측 확률
action_df['Actual_Churn'] = y_test.values

# 마케팅 액션 등급(Action_Grade) 생성
def assign_action_grade(row):
    prob = row['Churn_Prob']
    tenure = row['tenure']
    monthly_charges = row['MonthlyCharges']

    if prob >= 0.7:
        if monthly_charges > action_df['MonthlyCharges'].median():
            return '1_Emergency_VIP'  # 고액 결제 중인 초고위험군 (최우선 방어)
        return '2_High_Risk'           # 일반 초고위험군
    elif 0.4 <= prob < 0.7:
        if tenure > 24:
            return '3_Loyal_At_Risk'    # 장기 이용자 중 이탈 징후 (혜택 강화)
        return '4_Potential_Churn'      # 잠재적 이탈군
    else:
        return '5_Safe_Zone'            # 안정권

action_df['Action_Grade'] = action_df.apply(assign_action_grade, axis=1)

# 3. 등급별 대응 전략(Action_Strategy) 매핑
strategy_map = {
    '1_Emergency_VIP': '전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문)',
    '2_High_Risk': '단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사',
    '3_Loyal_At_Risk': '장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안',
    '4_Potential_Churn': '부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도',
    '5_Safe_Zone': '현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토'
}

action_df['Action_Strategy'] = action_df['Action_Grade'].map(strategy_map)

# 4. 결과 요약 통계 확인
action_summary = action_df.groupby('Action_Grade').agg({
    'Churn_Prob': 'mean',
    'MonthlyCharges': 'mean',
    'tenure': 'mean',
    'Action_Strategy': 'first',
    'Actual_Churn': 'count'
}).rename(columns={'Actual_Churn': 'Customer_Count'})

print("=== [고객 등급별 마케팅 액션 요약] ===")
display(action_summary.sort_index())

# 5. 실무용: 즉시 연락이 필요한 'Emergency_VIP' 명단 추출
emergency_list = action_df[action_df['Action_Grade'] == '1_Emergency_VIP'].sort_values(by='Churn_Prob', ascending=False)
print(f"\n[알림] 최우선 관리 대상(Emergency_VIP) {len(emergency_list)}명을 발견했습니다.")

=== [고객 등급별 마케팅 액션 요약] ===


,Churn_Prob,MonthlyCharges,tenure,Action_Strategy,Customer_Count
Action_Grade,,,,,
1_Emergency_VIP,0.822461,90.522794,35.926471,전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문),68
2_High_Risk,0.818548,38.984211,25.122807,단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사,57
3_Loyal_At_Risk,0.518520,69.233548,52.548387,장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안,155
4_Potential_Churn,0.524028,58.777027,9.333333,부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도,111
5_Safe_Zone,0.127491,63.524656,31.348723,현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토,1018



[알림] 최우선 관리 대상(Emergency_VIP) 68명을 발견했습니다.
